## Steps:
- Step 1 - Load and validate the data
- Step 2 - Investigate the constant feature
- Step 3 - Validate feature representations
- Step 4 - Convert Boolean features to integers
-  Step 5 - Decide how to represent `src_port` and `dst_port`
- Step 6 - Save the prepared notebook

## Step 01

In [10]:
!pip install -q "kagglehub[pandas-datasets]"

import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Download the dataset using KaggleHub
dataset_path = kagglehub.dataset_download(
    "ziya07/network-traffic-anomaly-detection-dataset"
)

# Inspect available files
print("Dataset directory:", dataset_path)

for file in Path(dataset_path).rglob("*"):
    if file.is_file():
        print(file.name)

csv_files = list(Path(dataset_path).rglob("*.csv"))

if len(csv_files) != 1:
    print("CSV files found:")
    for file in csv_files:
        print(file)
else:
    df = pd.read_csv(csv_files[0])

    print("Dataset shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())

Dataset directory: /root/.cache/kagglehub/datasets/ziya07/network-traffic-anomaly-detection-dataset/versions/1
embedded_system_network_security_dataset.csv
Dataset shape: (1000, 18)
Columns: ['packet_size', 'inter_arrival_time', 'src_port', 'dst_port', 'packet_count_5s', 'mean_packet_size', 'spectral_entropy', 'frequency_band_energy', 'label', 'protocol_type_TCP', 'protocol_type_UDP', 'src_ip_192.168.1.2', 'src_ip_192.168.1.3', 'dst_ip_192.168.1.5', 'dst_ip_192.168.1.6', 'tcp_flags_FIN', 'tcp_flags_SYN', 'tcp_flags_SYN-ACK']


,packet_size,inter_arrival_time,src_port,dst_port,packet_count_5s,mean_packet_size,spectral_entropy,frequency_band_energy,label,protocol_type_TCP,protocol_type_UDP,src_ip_192.168.1.2,src_ip_192.168.1.3,dst_ip_192.168.1.5,dst_ip_192.168.1.6,tcp_flags_FIN,tcp_flags_SYN,tcp_flags_SYN-ACK
0,0.405154,0.620362,62569,443,0.857143,0.0,0.834066,0.534891,0.0,False,True,True,False,False,False,False,False,False
1,0.527559,0.741288,59382,443,0.785714,0.0,0.147196,0.990757,0.0,False,True,False,False,False,True,False,True,False
2,0.226199,0.485116,65484,80,0.285714,0.0,0.855192,0.031781,0.0,False,True,False,False,True,False,False,False,False
3,0.573372,0.450965,51707,53,0.142857,0.0,0.153220,0.169958,0.0,False,False,False,True,False,False,False,False,False
4,0.651396,0.888740,26915,53,0.714286,0.0,0.923916,0.552053,0.0,True,False,False,True,False,False,False,True,False


In [11]:
df.head()

,packet_size,inter_arrival_time,src_port,dst_port,packet_count_5s,mean_packet_size,spectral_entropy,frequency_band_energy,label,protocol_type_TCP,protocol_type_UDP,src_ip_192.168.1.2,src_ip_192.168.1.3,dst_ip_192.168.1.5,dst_ip_192.168.1.6,tcp_flags_FIN,tcp_flags_SYN,tcp_flags_SYN-ACK
0,0.405154,0.620362,62569,443,0.857143,0.0,0.834066,0.534891,0.0,False,True,True,False,False,False,False,False,False
1,0.527559,0.741288,59382,443,0.785714,0.0,0.147196,0.990757,0.0,False,True,False,False,False,True,False,True,False
2,0.226199,0.485116,65484,80,0.285714,0.0,0.855192,0.031781,0.0,False,True,False,False,True,False,False,False,False
3,0.573372,0.450965,51707,53,0.142857,0.0,0.153220,0.169958,0.0,False,False,False,True,False,False,False,False,False
4,0.651396,0.888740,26915,53,0.714286,0.0,0.923916,0.552053,0.0,True,False,False,True,False,False,False,True,False


## Step 02

In [12]:
# Separate features and target

X = df.drop(columns=['label'])
y = df['label']

# Count the unique values in each feature
unique_counts = X.nunique(dropna=False).sort_values()
print("Features with only one unique value:")
display(unique_counts[unique_counts == 1])

# Check mean_packet_size specifically
print("\nmean_packet_size value counts:")
print(df["mean_packet_size"].value_counts(dropna=False))

Features with only one unique value:


,0
mean_packet_size,1



mean_packet_size value counts:
mean_packet_size
0.0    1000
Name: count, dtype: int64


`mean_packet_size` is zero in all 1,000 records. This feature has zero variance, so it cannot help distinguish normal traffic from malicious traffic.

In [13]:
X = X.drop(columns=['mean_packet_size'])

print("Original dataset shape:",df.shape)
print("Feature matrix shape:",X.shape)
print("Target shape:",y.shape)

Original dataset shape: (1000, 18)
Feature matrix shape: (1000, 16)
Target shape: (1000,)


## Step 03

In [18]:
print("Data types:")
print(X.dtypes)

print("\nNumerical feature ranges:")
display(X.select_dtypes(include="number").describe().T)

print("\nBoolean feature counts:")
print(X.select_dtypes(include='bool').nunique())

Data types:
packet_size              float64
inter_arrival_time       float64
src_port                   int64
dst_port                   int64
packet_count_5s          float64
spectral_entropy         float64
frequency_band_energy    float64
protocol_type_TCP           bool
protocol_type_UDP           bool
src_ip_192.168.1.2          bool
src_ip_192.168.1.3          bool
dst_ip_192.168.1.5          bool
dst_ip_192.168.1.6          bool
tcp_flags_FIN               bool
tcp_flags_SYN               bool
tcp_flags_SYN-ACK           bool
dtype: object

Numerical feature ranges:


,count,mean,std,min,25%,50%,75%,max
packet_size,1000.0,0.502446,0.289606,0.0,0.256263,0.499642,0.741410,1.0
inter_arrival_time,1000.0,0.512259,0.281130,0.0,0.275909,0.515971,0.746523,1.0
src_port,1000.0,32024.617000,18520.890349,1038.0,16245.250000,31883.000000,47746.250000,65484.0
dst_port,1000.0,199.769000,180.078488,53.0,53.000000,80.000000,443.000000,443.0
packet_count_5s,1000.0,0.506857,0.303271,0.0,0.267857,0.500000,0.785714,1.0
spectral_entropy,1000.0,0.495222,0.292927,0.0,0.236912,0.504290,0.761556,1.0
frequency_band_energy,1000.0,0.485651,0.295953,0.0,0.228039,0.467905,0.750876,1.0



Boolean feature counts:
protocol_type_TCP     2
protocol_type_UDP     2
src_ip_192.168.1.2    2
src_ip_192.168.1.3    2
dst_ip_192.168.1.5    2
dst_ip_192.168.1.6    2
tcp_flags_FIN         2
tcp_flags_SYN         2
tcp_flags_SYN-ACK     2
dtype: int64


- `src_port` is a port identifier, not an ordinary continous measurement.

In [19]:
X.head()

,packet_size,inter_arrival_time,src_port,dst_port,packet_count_5s,spectral_entropy,frequency_band_energy,protocol_type_TCP,protocol_type_UDP,src_ip_192.168.1.2,src_ip_192.168.1.3,dst_ip_192.168.1.5,dst_ip_192.168.1.6,tcp_flags_FIN,tcp_flags_SYN,tcp_flags_SYN-ACK
0,0.405154,0.620362,62569,443,0.857143,0.834066,0.534891,False,True,True,False,False,False,False,False,False
1,0.527559,0.741288,59382,443,0.785714,0.147196,0.990757,False,True,False,False,False,True,False,True,False
2,0.226199,0.485116,65484,80,0.285714,0.855192,0.031781,False,True,False,False,True,False,False,False,False
3,0.573372,0.450965,51707,53,0.142857,0.153220,0.169958,False,False,False,True,False,False,False,False,False
4,0.651396,0.888740,26915,53,0.714286,0.923916,0.552053,True,False,False,True,False,False,False,True,False


## Step 04

In [25]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=['mean_packet_size'])

bool_cols = df_clean.select_dtypes(include=['bool']).columns
df_clean[bool_cols] = df_clean[bool_cols].astype('int8')

print("\nUpdated data types:")
print(df_clean.dtypes)

print("\nDataset shape:",df_clean.shape)


Updated data types:
packet_size              float64
inter_arrival_time       float64
src_port                   int64
dst_port                   int64
packet_count_5s          float64
spectral_entropy         float64
frequency_band_energy    float64
label                    float64
protocol_type_TCP           int8
protocol_type_UDP           int8
src_ip_192.168.1.2          int8
src_ip_192.168.1.3          int8
dst_ip_192.168.1.5          int8
dst_ip_192.168.1.6          int8
tcp_flags_FIN               int8
tcp_flags_SYN               int8
tcp_flags_SYN-ACK           int8
dtype: object

Dataset shape: (1000, 17)


## Step 05

In [30]:
print("Destination port values:")
print(df_clean['dst_port'].value_counts().sort_index())

print("\nNumber of unique source ports:")
print(df_clean['src_port'].nunique())

print("\nSource port range:")
print(df_clean['src_port'].min(),"to",df_clean['src_port'].max())



Destination port values:
dst_port
53     310
80     337
443    353
Name: count, dtype: int64

Number of unique source ports:
995

Source port range:
1038 to 65484


- `dst_port` treate as categorical, the numerical distances between them aren't meaningful.

- `src_port` With 995 unique values, one-hot encoding would be impractical, retain it in clean dataset for now.

## Step 06

In [31]:
from pathlib import Path

# Create an output directory
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the prepared dataset
output_path = output_dir / "network_traffic_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("File path:", output_path)
print("Dataset shape:", df_clean.shape)

Dataset saved successfully!
File path: data/processed/network_traffic_cleaned.csv
Dataset shape: (1000, 17)
